In [1]:
# 📘 Notebook: Fine-tuning DistilBART with Chunking and BERTScore + ROUGE Evaluation

In [3]:
# 🧩 1. Install Required Libraries
!pip install -q transformers evaluate datasets bert-score sentencepiece rouge-score
%pip install transformers==4.54.0 accelerate datasets evaluate bert-score sentencepiece rouge_score
# stable version
%pip install transformers==4.38.2 accelerate datasets evaluate bert-score sentencepiece rouge_score

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement transformers==4.54.0 (from versions: 0.1, 2.0.0, 2.1.0, 2.1.1, 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 3.0.0, 3.0.1, 3.0.2, 3.1.0, 3.2.0, 3.3.0, 3.3.1, 3.4.0, 3.5.0, 3.5.1, 4.0.0rc1, 4.0.0, 4.0.1, 4.1.0, 4.1.1, 4.2.0, 4.2.1, 4.2.2, 4.3.0rc1, 4.3.0, 4.3.1, 4.3.2, 4.3.3, 4.4.0, 4.4.1, 4.4.2, 4.5.0, 4.5.1, 4.6.0, 4.6.1, 4.7.0, 4.8.0, 4.8.1, 4.8.2, 4.9.0, 4.9.1, 4.9.2, 4.10.0, 4.10.1, 4.10.2, 4.10.3, 4.11.0, 4.11.1, 4.11.2, 4.11.3, 4.12.0, 4.12.1, 4.12.2, 4.12.3, 4.12.4, 4.12.5, 4.13.0, 4.14.0, 4.14.1, 4.15.0, 4.16.0, 4.16.1, 4.16.2, 4.17.0, 4.18.0, 4.19.0, 4.19.1, 4.19.2, 4.19.3, 4.19.4, 4.20.0, 4.20.1, 4.21.0, 4.21.1, 4.21.2, 4.21.3, 4.22.0, 4.22.1, 4.22.2, 4.23.0, 4.23.1, 4.24.0, 4.25.0, 4.25.1, 4.26.0, 4.26.1, 4.27.0, 4.27.1, 4.27.2, 4.27.3, 4.27.4, 4.28.0, 4.28.1, 4.29.0, 4.29.1, 4.29.2, 4.30.0, 4.30.1, 4.30.2, 4.31.0, 4.32.0, 4.32.1, 4.33.0, 4.33.1, 4.33.2

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: requests in c:\users\user\anaconda3\envs\learn-env\lib\site-packages (from transformers==4.38.2) (2.32.4)


In [4]:
# 🧪 2. Import Libraries
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
import evaluate
from tqdm import tqdm
tqdm.pandas()

import os
# os.environ["WANDB_DISABLED"] = "true"

RuntimeError: Failed to import transformers.trainer_seq2seq because of the following error (look up to see its traceback):
Failed to import transformers.integrations.integration_utils because of the following error (look up to see its traceback):
cannot import name 'DEFAULT_CIPHERS' from 'urllib3.util.ssl_' (c:\Users\User\anaconda3\envs\learn-env\lib\site-packages\urllib3\util\ssl_.py)

In [61]:
# 📊 3. Load the Dataset
df = pd.read_csv("/kaggle/input/3cols-final-summary-dataset-gpt4o/fixed_final_summaries_4o.csv")
df = df.dropna(subset=["content", "summary"])

In [62]:
# 🧠 4. Initialize Tokenizer and Model
model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [63]:
# 🔪 5. Chunking Function (to handle long input content)
MAX_INPUT_TOKENS = 1024
CHUNK_OVERLAP = 50

In [64]:
def chunk_text(text, tokenizer, max_length=MAX_INPUT_TOKENS, overlap=CHUNK_OVERLAP):
    tokens = tokenizer.encode(text, truncation=False)
    chunks = []
    i = 0
    while i < len(tokens):
        chunk = tokens[i:i + max_length]
        decoded_chunk = tokenizer.decode(chunk, skip_special_tokens=True)
        chunks.append(decoded_chunk)
        i += max_length - overlap
    return chunks

In [65]:
# Apply chunking to content
df["chunked_inputs"] = df["content"].progress_apply(
    lambda x: chunk_text(x, tokenizer)
)

100%|██████████| 4035/4035 [00:07<00:00, 533.19it/s] 


In [66]:
# Flatten chunked data
flat_data = []
for _, row in df.iterrows():
    for chunk in row["chunked_inputs"]:
        flat_data.append({"input": chunk, "summary": row["summary"]})

In [67]:
flat_df = pd.DataFrame(flat_data)
hf_dataset = Dataset.from_pandas(flat_df)

In [68]:
# 🧪 Preprocessing Function
MAX_TARGET_LENGTH = 128
def preprocess(batch):
    inputs = tokenizer(batch["input"], truncation=True, padding="max_length", max_length=MAX_INPUT_TOKENS)
    targets = tokenizer(batch["summary"], truncation=True, padding="max_length", max_length=MAX_TARGET_LENGTH)
    inputs["labels"] = targets["input_ids"]
    return inputs

In [69]:
tokenized_ds = hf_dataset.map(preprocess, batched=True, remove_columns=hf_dataset.column_names)

Map:   0%|          | 0/5260 [00:00<?, ? examples/s]

In [71]:
# 🏋️ Training Arguments
# training_args = Seq2SeqTrainingArguments(
#     output_dir="outputs",
#     # evaluation_strategy="epoch",
#     learning_rate=3e-5,
#     per_device_train_batch_size=4,
#     per_device_eval_batch_size=4,
#     weight_decay=0.01,
#     save_total_limit=2,
#     num_train_epochs=3,
#     predict_with_generate=True,
#     fp16=torch.cuda.is_available(),
# )

training_args = Seq2SeqTrainingArguments(
    output_dir="outputs",
    # evaluation_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [72]:
# 🤖 Setup Trainer
data_collator = DataCollatorForSeq2Seq(tokenizer)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    eval_dataset=tokenized_ds.select(range(100)), #was previously 100
    # tokenizer=tokenizer,
    data_collator=data_collator,
)

In [73]:
# 🚀 Train the Fine-tuned Model
trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
500,2.299100
1000,1.833500
1500,1.629500


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3854: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarnin

TrainOutput(global_step=1974, training_loss=1.810921310533023, metrics={'train_runtime': 4244.6498, 'train_samples_per_second': 3.718, 'train_steps_per_second': 0.465, 'total_flos': 2.442605270925312e+16, 'train_loss': 1.810921310533023, 'epoch': 3.0})

In [74]:
# 🔍 Load Evaluation Metrics
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

In [75]:
# 📏 Helper Function to Evaluate Any Model
# def evaluate_model(model, tokenizer, dataset):
#     trainer = Seq2SeqTrainer(
#         model=model,
#         tokenizer=tokenizer,
#         data_collator=data_collator)
#     predictions = trainer.predict(dataset)
#     decoded_preds = tokenizer.batch_decode(predictions.predictions, skip_special_tokens=True)
#     decoded_labels = tokenizer.batch_decode(predictions.label_ids, skip_special_tokens=True)
#     # Compute ROUGE
#     rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
#     # Compute BERTScore
#     bert_result = bertscore.compute(predictions=decoded_preds, references=decoded_labels, lang="en")
        
#     bert_avg = {
#         "precision": round(sum(bert_result["precision"]) / len(bert_result["precision"]), 4),
#         "recall": round(sum(bert_result["recall"]) / len(bert_result["recall"]), 4),
#         "f1": round(sum(bert_result["f1"]) / len(bert_result["f1"]), 4) }
#     return rouge_result, bert_avg, decoded_preds, decoded_labels


def evaluate_model(model, tokenizer, dataset):
    eval_trainer = Seq2SeqTrainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=data_collator,
        args=training_args,
    )
    predictions = eval_trainer.predict(dataset)
    decoded_preds = tokenizer.batch_decode(predictions.predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(predictions.label_ids, skip_special_tokens=True)

    # Compute ROUGE
    rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    # Compute BERTScore
    bert_result = bertscore.compute(predictions=decoded_preds, references=decoded_labels, lang="en")

    bert_avg = {
        "precision": round(sum(bert_result["precision"]) / len(bert_result["precision"]), 4),
        "recall": round(sum(bert_result["recall"]) / len(bert_result["recall"]), 4),
        "f1": round(sum(bert_result["f1"]) / len(bert_result["f1"]), 4)
    }

    return rouge_result, bert_avg, decoded_preds, decoded_labels

In [76]:
# 🎯 Evaluate Baseline (Unfine-tuned)
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
baseline_rouge, baseline_bert, baseline_summaries, references = evaluate_model(baseline_model, tokenizer, tokenized_ds.select(range(100)))#tokenizer_ds.select(range(100))
print("\n📊 Baseline ROUGE:", baseline_rouge)
print("📊 Baseline BERTScore:", baseline_bert)


/tmp/ipykernel_36/1803282067.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  eval_trainer = Seq2SeqTrainer(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)



📊 Baseline ROUGE: {'rouge1': 0.29208831162610704, 'rouge2': 0.06298202050562325, 'rougeL': 0.17105664683138228, 'rougeLsum': 0.2009150160462877}
📊 Baseline BERTScore: {'precision': 0.85, 'recall': 0.8474, 'f1': 0.8487}


In [77]:
# 🎯 Evaluate Finetuned
finetuned_rouge, finetuned_bert, finetuned_summaries, _ = evaluate_model(model, tokenizer, tokenized_ds.select(range(100))) #range was 100 before
print("\n📊 Finetuned ROUGE:", finetuned_rouge)
print("📊 Finetuned BERTScore:", finetuned_bert)




/tmp/ipykernel_36/1803282067.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  eval_trainer = Seq2SeqTrainer(



📊 Finetuned ROUGE: {'rouge1': 0.5015079748985711, 'rouge2': 0.2009015794990769, 'rougeL': 0.30756104735837697, 'rougeLsum': 0.37530642725495517}
📊 Finetuned BERTScore: {'precision': 0.8921, 'recall': 0.8915, 'f1': 0.8917}


In [78]:
# 📝 Compare Summaries Side-by-Side
comparison_df = pd.DataFrame({
    "Input": flat_df.loc[:99, "input"].values,
    "Reference": references,
    "Baseline Summary": baseline_summaries,
    "Finetuned Summary": finetuned_summaries
})

In [79]:
# Display a few examples
comparison_df.sample(5, random_state=42)

,Input,Reference,Baseline Summary,Finetuned Summary
83,"b model, which is much more reasonable and fit...",Preference optimization is a method used to tr...,Our memory calculation isn't exact as it does...,This article explains how to train a large lan...
53,MotivationBackground on LoRAMulti-LoRA Serving...,The article discusses a new method called Mult...,Multi-LoRA serving is a technique to fine-tun...,The article discusses a new feature called Mul...
70,How the Reformer uses less than 8GB of RAM to ...,The Reformer is a new model designed to handle...,How the Reformer uses less than 8GB of RAM to...,The Reformer model is a powerful tool for hand...
45,"UB device radix sort, a highly optimized sort ...",3D Gaussian Splatting is a method for creating...,UB device radix sort is a highly optimized sor...,Hugging Face has introduced a new way to speed...
44,What is 3D Gaussian Splatting?How it works1. S...,3D Gaussian Splatting is a method for creating...,3D Gaussian Splatting allows real-time render...,3D Gaussian Splatting is a new method used to ...
